# DepthWizard demo-tuned training (stratified GAMUS subset, ~5-7 h GPU)

Trains the production multitask model (nDSM + 7-class semantics + building boundary)
with demo-prioritized loss weights, on a ~2,900-tile stratified subset of the full
GAMUS dataset, using the repository trainer (`kaggle_final_train.py` + DPT decoder).

**Before running, enable in the notebook settings:**
- Accelerator: **GPU (T4 or P100)**
- Internet: **ON** (needed to download Depth Anything V2 Base weights)
- Attach datasets: `aishwk/gamus-remote-sensing-dataset` (full GAMUS) and
  `kunalkokande/salmanbhai` (DepthWizard training code)

Flow: cells 1-7 set up and validate (~10 min) -> **Cell 8 is a full-pipeline smoke
gate on a 28-tile micro-subset (~10-15 min)** -> Cell 9 is the real training
(~4-6 h, epoch and step progress stream live) -> cells 10-12 verify and report.

If the smoke gate passes, the long run will not hit code errors - it executes the
identical code path (audit -> class weights -> epochs -> validation -> best
checkpoint -> test eval -> metrics -> production service load) at tiny scale.
Safe to "Save & Run All" for an unattended ~7 h session.
Output: a production `dinosaur.pth` that drops straight into the hackathon backend.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.45,<5", "accelerate", "h5py>=3.10,<4",
])
print("Dependencies installed")


In [ ]:
from pathlib import Path

import torch

CODE_ROOT = Path("/kaggle/input/datasets/kunalkokande/salmanbhai")
SOURCE_ROOT = Path("/kaggle/input/datasets/aishwk/gamus-remote-sensing-dataset")

SRC_ROOT = Path("/kaggle/working/depthwizard_src")
SUBSET_ROOT = Path("/kaggle/working/gamus_subset")
RUN_ROOT = Path("/kaggle/working/dinosaur_run")

MODEL_ID = "depth-anything/Depth-Anything-V2-Base-hf"
CROP_SIZE = 518

# If Cell 7 hits a CUDA out-of-memory error, set BATCH_SIZE = 2 and ACCUMULATION = 2
# and re-run from this cell.
BATCH_SIZE = 4
ACCUMULATION = 1
NUM_WORKERS = 4
EPOCHS = 10
CROPS_PER_SAMPLE = 2
SEED = 7

TARGETS = {"train": 2400, "val": 250, "test": 250}
DATASET_ID = "aishwk/gamus-remote-sensing-dataset"
DATASET_REVISION = "stratified-subset-2400-250-250-seed7"

assert torch.cuda.is_available(), "Enable the GPU accelerator (Settings -> Accelerator -> GPU)."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))


def resolve_gamus_root():
    candidates = [SOURCE_ROOT]
    if Path("/kaggle/input").exists():
        for path in Path("/kaggle/input").rglob("images/train"):
            if path.is_dir():
                candidates.append(path.parent.parent)
    for root in candidates:
        required = [root / kind / split for kind in ("images", "heights", "classes") for split in ("train", "val", "test")]
        if all(path.is_dir() for path in required):
            return root
    raise FileNotFoundError(
        "Could not find a GAMUS dataset (images/heights/classes x train/val/test) under /kaggle/input. "
        "Attach aishwk/gamus-remote-sensing-dataset to this notebook."
    )


SOURCE_ROOT = resolve_gamus_root()
if not CODE_ROOT.exists():
    raise FileNotFoundError(f"Code dataset not found at {CODE_ROOT}. Attach kunalkokande/salmanbhai.")

for directory in (SRC_ROOT, RUN_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

print("GAMUS root:", SOURCE_ROOT)
print("Code root:", CODE_ROOT)


In [ ]:
import shutil
import sys

REQUIRED_SOURCES = {
    "kaggle_final_train.py": "training",
    "depthwizard_model.py": "training",
    "gamus_audit.py": "training",
    "model_service.py": "app",
    "semantic_contract.py": "app",
}

(SRC_ROOT / "training").mkdir(parents=True, exist_ok=True)
(SRC_ROOT / "app").mkdir(parents=True, exist_ok=True)
(SRC_ROOT / "training" / "__init__.py").write_text("")
(SRC_ROOT / "app" / "__init__.py").write_text("")

for name, package in REQUIRED_SOURCES.items():
    matches = sorted(CODE_ROOT.rglob(name))
    if not matches:
        raise FileNotFoundError(f"Missing {name} below {CODE_ROOT}. Check the salmanbhai code dataset.")
    shutil.copy2(matches[0], SRC_ROOT / package / name)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from training.kaggle_final_train import BUILDING_CLASS, CLASS_COUNT, CLASS_NAMES

assert CLASS_COUNT == 7 and BUILDING_CLASS == 3
print("Training source ready:", SRC_ROOT)
print("Class contract:", list(CLASS_NAMES))


In [ ]:
import h5py

EXPECTED_KEYS = {"images": "image", "heights": "height", "classes": "classes"}
KEYS = {}

for kind in ("images", "heights", "classes"):
    files = sorted((SOURCE_ROOT / kind / "train").rglob("*.h5"))[:3]
    discovered = []
    for path in files:
        with h5py.File(path, "r") as handle:
            discovered.append([key for key in handle.keys() if isinstance(handle[key], h5py.Dataset)])
    keys = discovered[0]
    assert all(entry == keys for entry in discovered), f"Inconsistent HDF5 keys for {kind}: {discovered}"
    KEYS[kind] = EXPECTED_KEYS[kind] if EXPECTED_KEYS[kind] in keys else keys[0]
    print(f"{kind}: datasets={keys} -> using '{KEYS[kind]}'")


In [ ]:
import random
import shutil
from collections import defaultdict

from training.kaggle_final_train import paired_samples


def city_of(path):
    return path.name.split("_", 1)[0]


if SUBSET_ROOT.exists():
    shutil.rmtree(SUBSET_ROOT)

for split, target in TARGETS.items():
    triples = paired_samples(SOURCE_ROOT, split)
    by_city = defaultdict(list)
    for image, height, classes in triples:
        by_city[city_of(image)].append((image, height, classes))

    rng = random.Random(SEED)
    selected = []
    for city in sorted(by_city):
        files = sorted(by_city[city])
        share = max(1, min(len(files), round(target * len(files) / len(triples))))
        selected.extend(rng.sample(files, share))
    rng.shuffle(selected)
    selected = selected[:target]

    for image, height, classes in selected:
        for source, kind in ((image, "images"), (height, "heights"), (classes, "classes")):
            directory = SUBSET_ROOT / kind / split
            directory.mkdir(parents=True, exist_ok=True)
            destination = directory / source.name
            if not destination.exists():
                destination.symlink_to(source)

    distribution = ", ".join(f"{city}:{len(by_city[city])}" for city in sorted(by_city))
    print(f"{split}: {len(selected)} tiles | cities available -> {distribution}")

print("\nSubset totals (images/heights/classes):")
for split in TARGETS:
    counts = {kind: len(list((SUBSET_ROOT / kind / split).glob("*.h5"))) for kind in ("images", "heights", "classes")}
    assert counts["images"] == counts["heights"] == counts["classes"], f"Unpaired files in {split}: {counts}"
    print(split, counts)


In [ ]:
from pathlib import Path
import sys

TRAINER = SRC_ROOT / "training" / "kaggle_final_train.py"
source = TRAINER.read_text()

# Patch 1: demo-prioritized task weights.
# Stock: semantic 0.15 / boundary 0.05, ramped to full strength only by epoch 7.
# Demo:  semantic 0.30 / boundary 0.15, ramped over the first 2 epochs so both
# heads train for almost the whole 10-epoch run.
old_weights = '''
        semantic_weight = 0.15 * min(1.0, max(0.0, (epoch - 2) / 5.0))
        boundary_weight = 0.05 * min(1.0, max(0.0, (epoch - 2) / 5.0))'''
new_weights = '''
        semantic_weight = 0.30 * min(1.0, max(0.0, epoch / 2.0))
        boundary_weight = 0.15 * min(1.0, max(0.0, epoch / 2.0))'''
assert old_weights in source, "Loss-weight lines not found; the salmanbhai trainer differs from the repo"
source = source.replace(old_weights, new_weights, 1)

# Patch 2: unfreeze the Depth Anything backbone from epoch 1 instead of epoch 3,
# so the backbone adapts during most of this short run.
old_freeze = "        if epoch == 3:"
new_freeze = "        if epoch == 1:"
assert old_freeze in source, "Unfreeze line not found; the salmanbhai trainer differs from the repo"
source = source.replace(old_freeze, new_freeze, 1)

# Patch 3: live step-progress logging inside every epoch, so the long run is
# visibly alive (one line every 100 steps) instead of silent for ~20 min/epoch.
old_progress = '''
    for step, batch in enumerate(loader):
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}'''
new_progress = '''
    for step, batch in enumerate(loader):
        if step % 100 == 0:
            print(f"[epoch {epoch + 1}] step {step}/{len(loader)}", flush=True)
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}'''
assert old_progress in source, "Train-loop lines not found; the salmanbhai trainer differs from the repo"
source = source.replace(old_progress, new_progress, 1)

TRAINER.write_text(source)
sys.modules.pop("training.kaggle_final_train", None)
print("Patched trainer: semantic 0.30 / boundary 0.15 (2-epoch ramp), backbone unfrozen at epoch 1, step progress every 100 steps")


In [ ]:
import gc

import numpy as np
import torch
from torch.utils.data import DataLoader

from training.kaggle_final_train import GamusCropDataset, index_samples, load_triplet, multitask_loss
from training.depthwizard_model import build_depthwizard_model

torch.manual_seed(SEED)
device = torch.device("cuda")
keys = {"image": KEYS["images"], "height": KEYS["heights"], "classes": KEYS["classes"]}

samples = index_samples(SUBSET_ROOT)
print({split: len(values) for split, values in samples.items()})

# Quick height-scale estimate; the trainer computes its own from the full subset.
height_values = []
for sample in samples["train"][:24]:
    _, height, _, _ = load_triplet(sample, keys)
    finite = height[np.isfinite(height)]
    if finite.size:
        stride = max(1, finite.size // 4096)
        height_values.append(finite[::stride])
probe_height_scale = max(1.0, float(np.percentile(np.concatenate(height_values), 99.5)))
print("Probe height_scale estimate:", round(probe_height_scale, 2))

probe_dataset = GamusCropDataset(
    samples["train"][:BATCH_SIZE], keys, CROP_SIZE, probe_height_scale,
    np.ones(7, dtype=np.float32), True, 1, SEED,
)
probe_batch = next(iter(DataLoader(probe_dataset, batch_size=BATCH_SIZE, num_workers=0)))

model = build_depthwizard_model(MODEL_ID).to(device)
model.train()
probe_batch = {key: value.to(device) for key, value in probe_batch.items()}
outputs = model(probe_batch["image"])
losses = multitask_loss(outputs, probe_batch, torch.ones(7, device=device), probe_height_scale, 0.30, 0.15)
assert torch.isfinite(losses["total"]).item(), losses
losses["total"].backward()
torch.cuda.synchronize()

print("Losses:", {key: round(float(value.detach()), 4) for key, value in losses.items()})
print("Peak VRAM: %.1f / %.1f GB" % (
    torch.cuda.max_memory_allocated() / 1024**3,
    torch.cuda.get_device_properties(0).total_memory / 1024**3,
))

del model, outputs, losses, probe_batch
gc.collect()
torch.cuda.empty_cache()
print("Probe passed: batch size", BATCH_SIZE, "fits in VRAM")


In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch

SMOKE_SUBSET = Path("/kaggle/working/gamus_smoke_subset")
SMOKE_ROOT = Path("/kaggle/working/smoke_run")
SMOKE_TILES = {"train": 16, "val": 6, "test": 6}

assert SUBSET_ROOT.exists(), "Run the subset cell (Cell 5) first."

if SMOKE_SUBSET.exists():
    shutil.rmtree(SMOKE_SUBSET)
for split, count in SMOKE_TILES.items():
    for image in sorted((SUBSET_ROOT / "images" / split).glob("*.h5"))[:count]:
        for kind in ("images", "heights", "classes"):
            source = (SUBSET_ROOT / kind / split / image.name).resolve()
            directory = SMOKE_SUBSET / kind / split
            directory.mkdir(parents=True, exist_ok=True)
            destination = directory / image.name
            if not destination.exists():
                destination.symlink_to(source)

command = [
    sys.executable,
    str(SRC_ROOT / "training" / "kaggle_final_train.py"),
    "--root", str(SMOKE_SUBSET),
    "--output-dir", str(SMOKE_ROOT),
    "--dataset-id", DATASET_ID,
    "--dataset-revision", "smoke-gate",
    "--image-key", KEYS["images"],
    "--height-key", KEYS["heights"],
    "--class-key", KEYS["classes"],
    "--model-id", MODEL_ID,
    "--crop-size", str(CROP_SIZE),
    "--batch-size", str(BATCH_SIZE),
    "--accumulation", str(ACCUMULATION),
    "--num-workers", str(NUM_WORKERS),
    "--epochs", "2",
    "--crops-per-sample", "1",
    "--seed", str(SEED),
]
env = os.environ.copy()
env["PYTHONPATH"] = str(SRC_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTHONUNBUFFERED"] = "1"

started = time.time()
subprocess.run(command, check=True, env=env, cwd=SRC_ROOT)
elapsed = time.time() - started

history = json.loads((SMOKE_ROOT / "training_history.json").read_text())
for entry in history:
    print(json.dumps({
        "epoch": entry["epoch"],
        "train_loss_total": round(entry["losses"]["total"], 4),
        "val_macro_rmse": entry["validation"].get("macro_rmse"),
        "val_macro_semantic_miou": entry["validation"].get("macro_semantic_miou"),
        "seconds": round(entry["seconds"], 1),
    }))

smoke_checkpoint = torch.load(SMOKE_ROOT / "dinosaur.pth", map_location="cpu", weights_only=False)
assert smoke_checkpoint["checkpoint_format"] == "depthwizard_dpt_multiscale_v2"
assert smoke_checkpoint["class_count"] == 7 and smoke_checkpoint["building_class"] == 3
assert float(smoke_checkpoint["height_scale"]) >= 1.0
assert any(key.startswith("boundary_head") for key in smoke_checkpoint["state_dict"])
smoke_metrics = json.loads((SMOKE_ROOT / "metrics.json").read_text())
assert smoke_metrics["train_samples"] == SMOKE_TILES["train"]
assert smoke_metrics["validation_samples"] == SMOKE_TILES["val"]
assert smoke_metrics["test_samples"] == SMOKE_TILES["test"]

from app.model_service import DepthAnythingModelService

service = DepthAnythingModelService(SMOKE_ROOT / "dinosaur.pth", device="cuda")
from PIL import Image

probe_image = Image.fromarray(np.zeros((CROP_SIZE, CROP_SIZE, 3), dtype=np.uint8), mode="RGB")
probe_result = service.predict_result(probe_image)
assert probe_result["height"].shape == (CROP_SIZE, CROP_SIZE)
assert probe_result["semantic"].shape == (7, CROP_SIZE, CROP_SIZE)
del service, smoke_checkpoint
gc.collect()
torch.cuda.empty_cache()

print(f"\nSmoke gate PASSED in {elapsed / 60:.1f} min: full pipeline (audit -> epochs -> "
      f"best checkpoint -> test eval -> metrics -> production load) ran without errors.")
print(f"The real run trains on {TARGETS['train']} tiles for {EPOCHS} epochs - expect ~4-6 h.")
print("Do NOT download smoke_run/dinosaur.pth; it is a throwaway checkpoint.")


**Run the smoke gate above first (~10-15 min).** If it passed, this cell is safe to
run unattended. It is the long one: **~4-6 hours.** Output streams live: one
`[epoch N] step X/Y` line every 100 steps, and one JSON line per epoch with train
losses and validation metrics. Expect the first epoch JSON after ~30-40 min
(audit + class weights + epoch 1 + validation).


In [ ]:
import os
import subprocess
import sys
import time

command = [
    sys.executable,
    str(SRC_ROOT / "training" / "kaggle_final_train.py"),
    "--root", str(SUBSET_ROOT),
    "--output-dir", str(RUN_ROOT),
    "--dataset-id", DATASET_ID,
    "--dataset-revision", DATASET_REVISION,
    "--image-key", KEYS["images"],
    "--height-key", KEYS["heights"],
    "--class-key", KEYS["classes"],
    "--model-id", MODEL_ID,
    "--crop-size", str(CROP_SIZE),
    "--batch-size", str(BATCH_SIZE),
    "--accumulation", str(ACCUMULATION),
    "--num-workers", str(NUM_WORKERS),
    "--epochs", str(EPOCHS),
    "--crops-per-sample", str(CROPS_PER_SAMPLE),
    "--seed", str(SEED),
]

env = os.environ.copy()
env["PYTHONPATH"] = str(SRC_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTHONUNBUFFERED"] = "1"

print(f"Starting training: {EPOCHS} epochs x {TARGETS['train']} tiles x {CROPS_PER_SAMPLE} crops, batch {BATCH_SIZE}", flush=True)
started = time.time()
subprocess.run(command, check=True, env=env, cwd=SRC_ROOT)
print("\nTraining completed in %.1f h" % ((time.time() - started) / 3600))


In [ ]:
import json
import shutil

import torch

checkpoint_path = RUN_ROOT / "dinosaur.pth"
assert checkpoint_path.exists(), checkpoint_path
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

assert checkpoint["checkpoint_format"] == "depthwizard_dpt_multiscale_v2"
assert checkpoint["class_count"] == 7
assert checkpoint["building_class"] == 3
assert checkpoint["crop_size"] == CROP_SIZE
assert float(checkpoint["height_scale"]) >= 1.0
assert "decoder" in " ".join(checkpoint["state_dict"])
assert any(key.startswith("boundary_head") for key in checkpoint["state_dict"])

production = Path("/kaggle/working/dinosaur.pth")
shutil.copy2(checkpoint_path, production)

metrics = json.loads((RUN_ROOT / "metrics.json").read_text())
summary_keys = (
    "macro_rmse", "macro_mae", "macro_correlation", "macro_semantic_miou",
    "macro_building_iou", "macro_building_boundary_f1", "macro_boundary_f1",
)
print(json.dumps({
    "production_checkpoint": str(production),
    "size_mb": round(production.stat().st_size / 1024**2, 2),
    "height_scale": checkpoint["height_scale"],
    "validation": {key: metrics["validation"].get(key) for key in summary_keys},
    "test": {key: metrics["test"].get(key) for key in summary_keys},
}, indent=2))


In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image

from app.model_service import DepthAnythingModelService

service = DepthAnythingModelService(Path("/kaggle/working/dinosaur.pth"), device="cuda")
image = Image.fromarray(np.zeros((1024, 1024, 3), dtype=np.uint8), mode="RGB")
result = service.predict_result(image)

assert result["height"].shape == (1024, 1024)
assert result["semantic"].shape == (7, 1024, 1024)
assert np.isfinite(result["height"]).all()
assert np.isfinite(result["semantic"]).all()
semantic_ids = result["semantic"].argmax(axis=0)
assert semantic_ids.min() >= 0 and semantic_ids.max() <= 6
print({
    "checkpoint_load": "passed",
    "height_shape": result["height"].shape,
    "semantic_shape": result["semantic"].shape,
    "height_std": float(result["height"].std()),
})


In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import torch

from training.kaggle_final_train import BUILDING_CLASS, CLASS_NAMES, building_boundary, index_samples, load_triplet, tiled_predict
from training.depthwizard_model import build_depthwizard_model

class_names = list(CLASS_NAMES)
keys = {"image": KEYS["images"], "height": KEYS["heights"], "classes": KEYS["classes"]}
samples = index_samples(SUBSET_ROOT)

evaluation_model = build_depthwizard_model(MODEL_ID).to(device)
checkpoint = torch.load("/kaggle/working/dinosaur.pth", map_location=device, weights_only=False)
evaluation_model.load_state_dict(checkpoint["state_dict"], strict=True)
height_scale = float(checkpoint["height_scale"])


def iou(predicted, target):
    union = np.sum(predicted | target)
    return float(np.sum(predicted & target) / union) if union else None


def f1(predicted, target):
    denominator = np.sum(predicted) + np.sum(target)
    return float(2 * np.sum(predicted & target) / denominator) if denominator else 0.0


def per_class_iou(predicted, target, valid):
    values = {}
    for class_id, name in enumerate(class_names):
        p = (predicted == class_id) & valid
        t = (target == class_id) & valid
        union = np.sum(p | t)
        values[name] = float(np.sum(p & t) / union) if union else None
    return values


report_root = RUN_ROOT / "demo_report"
report_root.mkdir(parents=True, exist_ok=True)
records = []
for index, sample in enumerate(samples["val"][:6]):
    image, target_height, target_classes, valid = load_triplet(sample, keys)
    predicted_height, predicted_semantic, predicted_boundary = tiled_predict(
        evaluation_model, image, device, CROP_SIZE, 0.5, height_scale,
    )
    predicted_classes = predicted_semantic.argmax(axis=0).astype(np.int64)
    valid = valid.astype(bool)
    pred_building = (predicted_classes == BUILDING_CLASS) & valid
    true_building = (target_classes == BUILDING_CLASS) & valid
    difference = (predicted_height - target_height)[valid]
    records.append({
        "height_rmse": float(np.sqrt(np.mean(difference ** 2))) if difference.size else None,
        "height_mae": float(np.mean(np.abs(difference))) if difference.size else None,
        "building_iou": iou(pred_building, true_building),
        "building_f1": f1(pred_building, true_building),
        "boundary_f1": f1(predicted_boundary >= 0.0, building_boundary(target_classes) & valid),
        "per_class_iou": per_class_iou(predicted_classes, target_classes, valid),
    })

    if index < 4:
        figure, axes = plt.subplots(2, 4, figsize=(18, 9))
        panels = [
            (image, "RGB", None),
            (target_height, "Reference nDSM", "magma"),
            (predicted_height, "Predicted nDSM", "magma"),
            (np.abs(predicted_height - target_height), "Height error", "inferno"),
            (target_classes, "Reference classes", "tab10"),
            (predicted_classes, "Predicted classes", "tab10"),
            (true_building, "Reference buildings", "gray"),
            (pred_building, "Predicted buildings", "gray"),
        ]
        for axis, (values, title, cmap) in zip(axes.reshape(-1), panels):
            axis.imshow(values, cmap=cmap, vmin=0 if cmap == "tab10" else None, vmax=6 if cmap == "tab10" else None)
            axis.set_title(title)
            axis.axis("off")
        figure.suptitle(sample.image.name)
        figure.tight_layout()
        figure.savefig(report_root / f"{index:02d}_{sample.image.stem}.png", dpi=130)
        plt.show()

metric_names = ("height_rmse", "height_mae", "building_iou", "building_f1", "boundary_f1")
summary = {
    name: float(np.mean([record[name] for record in records if record[name] is not None]))
    for name in metric_names
}
summary["mean_per_class_iou"] = {
    name: float(np.mean([record["per_class_iou"][name] for record in records if record["per_class_iou"][name] is not None]))
    for name in class_names
}
print(json.dumps(summary, indent=2))


## Done - what to download and how to wire it up

Download from the Kaggle Output sidebar:

| File | Purpose |
|---|---|
| `dinosaur.pth` | Production checkpoint - place at the repo root, or set `DEPTHWIZARD_CHECKPOINT` |
| `metrics.json` | Validation/test metrics for the report |
| `demo_report/` | Side-by-side visual grids (reference vs predicted) |
| `dinosaur_run/validation_visuals/`, `dinosaur_run/test_visuals/` | Per-epoch visuals |

The checkpoint carries the full production contract
(`depthwizard_dpt_multiscale_v2`), so the FastAPI backend loads it through
`DepthAnythingModelService` with no code changes - building footprints and the
semantic viewer work out of the box.
